# Window Date Spans

Actual `create_time` span of each video window (`pre` / `cancel` / `post`) per case.

**Sources:** `data/<case>/<case>_youtube.jsonl` (`record_type == "video"`), cross-checked against `data/input/<case>.csv`.
Output matches the original dataset query: summary table (`case, window, n, min, max, span_days`) plus per-video detail listing.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd


In [ ]:
# Resolve repo root so the notebook runs from any cwd inside the repo
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "data").exists() and (REPO_ROOT / "02-preprocessing").exists():
        break
    REPO_ROOT = REPO_ROOT.parent

# Cases and canonical window order
CASES: list[str] = ["bodeoum_tv", "mimiminu"]
WINDOWS: list[str] = ["pre", "cancel", "post"]

DATA_ROOT = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "02-preprocessing" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Cases:", CASES)


## Load video metadata

Parse each `<case>_youtube.jsonl`, keep video records only with their upload `create_time` and `window_label`.

In [ ]:
def load_videos(case: str) -> pd.DataFrame:
    """Load video records for one case from its YouTube JSONL."""
    path = DATA_ROOT / case / f"{case}_youtube.jsonl"
    rows: list[dict[str, str]] = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if record.get("record_type") != "video":
                continue
            rows.append(
                {
                    "case": case,
                    "window": record.get("window_label"),
                    "video_id": record.get("video_id"),
                    "title": record.get("title"),
                    "create_time": record.get("create_time"),
                }
            )
    df = pd.DataFrame(rows)
    df["create_date"] = pd.to_datetime(df["create_time"], utc=True).dt.date
    return df.sort_values("create_time", kind="stable").reset_index(drop=True)


# Load all cases and concatenate — raw collection data is never modified here
videos_df = pd.concat([load_videos(case) for case in CASES], ignore_index=True)
videos_df.head()


## Window spans

One row per case × window: video count, earliest/latest upload date, and span in days.

In [ ]:
# Aggregate min/max upload date and day-span per case x window
spans_df = (
    videos_df.groupby(["case", "window"], as_index=False)
    .agg(n=("video_id", "size"), min=("create_date", "min"), max=("create_date", "max"))
)
spans_df["span_days"] = (pd.to_datetime(spans_df["max"]) - pd.to_datetime(spans_df["min"])).dt.days
spans_df["window"] = pd.Categorical(spans_df["window"], categories=WINDOWS, ordered=True)
spans_df = spans_df.sort_values(["case", "window"], kind="stable").reset_index(drop=True)
spans_df


## Per-video detail

Full date-ordered listing behind each span row — same detail as the original query.

In [ ]:
# Date-ordered detail view backing the span rows (pre -> cancel -> post)
_order = {w: i for i, w in enumerate(WINDOWS)}
details_df = (
    videos_df.assign(_worder=videos_df["window"].map(_order))
    .sort_values(["case", "_worder", "create_time"], kind="stable")
    .drop(columns="_worder")
    .reset_index(drop=True)[["case", "window", "create_date", "video_id", "title"]]
)
details_df


## Save outputs

In [ ]:
# Persist query-matching tables for downstream write-ups
spans_path = OUTPUT_DIR / "window_spans.csv"
details_path = OUTPUT_DIR / "window_span_details.csv"
spans_df.to_csv(spans_path, index=False)
details_df.to_csv(details_path, index=False)
print("Saved:", spans_path)
print("Saved:", details_path)


## Findings

- Each case holds 10 `pre` + 1 `cancel` + 10 `post` videos, matching `data/input/*.csv`.
- `bodeoum_tv`: pre spans 42 days (2024-04-06 → 2024-05-18); post spans 56 days (2024-07-06 → 2024-08-31).
- `mimiminu`: pre spans 12 days (2025-11-10 → 2025-11-22); post spans 17 days (2025-11-25 → 2025-12-12).
- Asymmetry to note downstream: bodeoum_tv windows cover ~2x the calendar time of mimiminu windows.